# Find Products Nearby With Qdrant Score Boosting

A shopper searches for **writing desk** and wants to collect it locally. A strong text match might be out of stock or too far away. Among eligible products, free returns and a shorter pickup distance can make one offer more attractive than another.

You will build semantic retrieval → stock/radius filters → a free-returns bonus → geographic decay → a combined ranking. You will inspect score contributions, compare the order against human product-relevance judgments, and see why excessive boosts and narrow candidate pools deserve scrutiny.

The product names, descriptions, query, and relevance labels come from [Wayfair's WANDS](https://github.com/wayfair/WANDS). **Pickup coordinates, stock, and free returns are deterministic teaching data**, not Wayfair locations, inventory, or policies. The notebook indexes all 1,123 products judged for the chosen query, rather than the whole catalog. That keeps CPU embedding practical while preserving a fully judged pool.

Allow **about 50 minutes**, including experiments; first-run downloads vary. Use a standard CPU runtime in Colab. No Qdrant account, API key, GPU, paid inference service, or Docker is required. This starts a real Qdrant server as a subprocess. The duration is an estimate, not a timed learner study.

## 1. Install the Pinned Dependencies

Use Qdrant server **1.20.2**, Python client **1.20.0**, and FastEmbed **0.8.0**. Formula queries require Qdrant 1.14 or later. Run the cells in order. If this kernel has already imported other versions, restart after installation.

In [1]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "qdrant-client==1.20.0",
        "fastembed==0.8.0",
        "pandas==2.3.3",
        "huggingface-hub==1.33.0",
        "httpx==0.28.1",
    ]
)

Out[1]: 0


## 2. Start an Isolated Qdrant Server

Download an official server release and verify its SHA-256. The bootstrap supports Linux x86-64 (the Colab path), macOS Apple Silicon, and macOS Intel. This run gets a temporary directory and a free localhost port. No existing Qdrant collection is touched.

In [2]:
import atexit
import asyncio
import hashlib
import platform
import shutil
import socket
import subprocess
import tarfile
import tempfile
from pathlib import Path

import httpx
from qdrant_client import AsyncQdrantClient, models

QDRANT_VERSION = "1.20.2"
RELEASES = {
    ("Linux", "x86_64"): (
        "qdrant-x86_64-unknown-linux-musl.tar.gz",
        "7c3e956b97830a9faa6391540330380c2fa4aa2f92df99bd8d1880a703b922af",
    ),
    ("Darwin", "arm64"): (
        "qdrant-aarch64-apple-darwin.tar.gz",
        "385b244e6283968b2be926358541b5bc19b9ada7ea9586e900684388bc912d5f",
    ),
    ("Darwin", "x86_64"): (
        "qdrant-x86_64-apple-darwin.tar.gz",
        "73a6538ccccb892ec2c6fa3431faf1e73fba7eb2179079adbd0c65ae4c55bf6a",
    ),
}
asset, expected_digest = RELEASES[(platform.system(), platform.machine())]
runtime_dir = Path(tempfile.mkdtemp(prefix="ecommerce-qdrant-"))
archive = runtime_dir / asset
release_url = (
    f"https://github.com/qdrant/qdrant/releases/download/v{QDRANT_VERSION}/{asset}"
)
async with httpx.AsyncClient(follow_redirects=True, timeout=120) as http:
    response = await http.get(release_url)
    response.raise_for_status()
    archive.write_bytes(response.content)
assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected_digest
binary = runtime_dir / "qdrant"
with tarfile.open(archive) as bundle:
    executable = bundle.extractfile("qdrant")
    assert executable is not None
    with binary.open("wb") as output:
        shutil.copyfileobj(executable, output)
binary.chmod(0o755)

with socket.socket() as port_probe:
    port_probe.bind(("127.0.0.1", 0))
    port = port_probe.getsockname()[1]
QDRANT_URL = f"http://127.0.0.1:{port}"
config = runtime_dir / "config.yaml"
config.write_text(
    "log_level: ERROR\n"
    "storage:\n  storage_path: ./storage\n  snapshots_path: ./snapshots\n"
    f"service:\n  host: 127.0.0.1\n  http_port: {port}\n  grpc_port: null\n"
    "  enable_cors: false\ncluster:\n  enabled: false\n"
)
server_log = (runtime_dir / "server.log").open("w")
server = subprocess.Popen(
    [str(binary), "--config-path", str(config), "--disable-telemetry"],
    cwd=runtime_dir,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
atexit.register(server.terminate)
async with httpx.AsyncClient(timeout=2) as http:
    for _ in range(60):
        if server.poll() is not None:
            raise RuntimeError((runtime_dir / "server.log").read_text())
        try:
            response = await http.get(f"{QDRANT_URL}/readyz")
            if response.is_success:
                break
        except httpx.ConnectError:
            pass
        await asyncio.sleep(0.5)
    else:
        raise TimeoutError("Qdrant did not become ready; inspect server.log.")
client = AsyncQdrantClient(url=QDRANT_URL)
server_info = await client.info()
assert server_info.version == QDRANT_VERSION
print(f"Qdrant server {server_info.version} is ready at {QDRANT_URL}")

Qdrant server 1.20.2 is ready at http://127.0.0.1:52751


## 3. Load a Judged WANDS Product Pool

WANDS contains 42,994 products, 480 queries, and 233,448 human-labeled query/product pairs. Its files have a `.csv` extension but use tabs. We download about 97 MB from a pinned repository revision and verify checksums. WANDS is released under the [MIT license](https://github.com/wayfair/WANDS/blob/3b74dcf4ba29ab8ff3e6a50b5b09fc627cb882b5/LICENSE).

Set `QUERY_ID=418` to use the original query **writing desk** and every product judged for it: 608 Exact, 390 Partial, and 125 Irrelevant. One duplicated judgment agrees on its label and is removed. We select by query ID, never by relevance grade. Grades stay outside the embedding and Qdrant payload; they are used only for the later relevance audit.

We add one simulated pickup offer per product. Product-ID arithmetic assigns stock, free returns, and one of eight fictional pickup coordinates around Warsaw. Some offers deliberately lack coordinates. These attributes are assigned without consulting the WANDS judgments and do not represent Wayfair operations. In a real marketplace, join maintained offer data to each product; multiple offers need distinct offer IDs and product-level deduplication.

For another query with a substantial judged pool, try `QUERY_ID=34` (bar stool with backrest), `124` (unique coffee tables), or `425` (side table with storage), and rerun this cell and every later cell. The query text, pool, embeddings, and judgment map must change together. This exercise expects at least five eligible products; some other WANDS queries have only one or two judgments.

In [3]:
import io
from typing import NotRequired, TypedDict

import pandas as pd
from IPython.display import display

WANDS_REVISION = "3b74dcf4ba29ab8ff3e6a50b5b09fc627cb882b5"
WANDS_BASE = f"https://raw.githubusercontent.com/wayfair/WANDS/{WANDS_REVISION}/dataset"
WANDS_SHA256 = {
    "product": "d993926254572e6eba96c8fd87cc549a17fb91ad3748308036eee4cf92b10ac6",
    "query": "63b61660560fecc33ec490804c7e2b81402ee3e7c31a9cbb5e03736639f68e95",
    "label": "c11fe81ad62f17f56f316b0ec9630ebe8fbe1393578cb0ca4f05c17253a180ef",
}


async def load_wands_file(name: str) -> pd.DataFrame:
    async with httpx.AsyncClient(follow_redirects=True, timeout=180) as http:
        response = await http.get(f"{WANDS_BASE}/{name}.csv")
        response.raise_for_status()
    assert hashlib.sha256(response.content).hexdigest() == WANDS_SHA256[name]
    return pd.read_csv(io.BytesIO(response.content), sep="\t", keep_default_na=False)


catalog, queries, labels = await asyncio.gather(
    *(load_wands_file(name) for name in ("product", "query", "label"))
)
QUERY_ID = 418
QUERY_TEXT = str(queries.loc[queries["query_id"] == QUERY_ID, "query"].iloc[0])
judgments = labels.loc[labels["query_id"] == QUERY_ID].copy()
assert judgments.groupby("product_id")["label"].nunique().max() == 1
judgments = judgments.drop_duplicates("product_id")
assert judgments["product_id"].is_unique
label_by_id = dict(zip(judgments["product_id"], judgments["label"], strict=True))
selected = catalog.loc[catalog["product_id"].isin(label_by_id)].sort_values(
    "product_id"
)
assert len(selected) == len(judgments)


class Product(TypedDict):
    id: int
    product_name: str
    product_description: str
    in_stock: bool
    free_returns: bool
    pickup_location: NotRequired[dict[str, float]]


PICKUP_LOCATIONS = [
    {"lat": 52.233, "lon": 21.020},
    {"lat": 52.238, "lon": 21.035},
    {"lat": 52.247, "lon": 20.985},
    {"lat": 52.258, "lon": 21.096},
    {"lat": 52.312, "lon": 21.043},
    {"lat": 52.126, "lon": 21.034},
    {"lat": 52.409, "lon": 21.014},
    {"lat": 52.227, "lon": 20.909},
]
products: list[Product] = []
for row in selected.to_dict(orient="records"):
    product_id = int(row["product_id"])
    product: Product = {
        "id": product_id,
        "product_name": str(row["product_name"]),
        "product_description": str(row["product_description"]),
        "in_stock": product_id % 7 != 0,
        "free_returns": product_id % 3 == 0,
    }
    if product_id % 37 != 0:
        product["pickup_location"] = PICKUP_LOCATIONS[
            product_id % len(PICKUP_LOCATIONS)
        ]
    products.append(product)
print(f"Query {QUERY_ID}: {QUERY_TEXT!r}; {len(products)} judged products")
display(judgments["label"].value_counts().rename_axis("label").to_frame("products"))

Query 418: 'writing desk'; 1123 judged products


,products
label,
Exact,608
Partial,390
Irrelevant,125


## 4. Embed Product Text and Create the Collection

FastEmbed embeds each product's original name and description with a pinned BGE small English v1.5 ONNX revision on CPU. It uses `passage_embed` for products and `query_embed` for the WANDS query. Batches of 32 limit peak memory use. The model truncates text beyond its context window. Operational fields and relevance labels never enter the embedding text.

Create a cosine collection, index the two Boolean fields and the geographic field, then ingest points using the original WANDS product IDs. Repeating this cell replaces only this notebook's collection in its isolated server.

In [4]:
from fastembed import TextEmbedding
from huggingface_hub import snapshot_download

MODEL_NAME = "BAAI/bge-small-en-v1.5"
MODEL_REVISION = "aa8f8b060edb00e03bfdd08813a2949946c8ba55"
model_path = snapshot_download(
    repo_id="qdrant/bge-small-en-v1.5-onnx-q",
    revision=MODEL_REVISION,
    allow_patterns=["*.json", "*.txt", "model_optimized.onnx"],
    cache_dir=str(runtime_dir / "model-cache"),
)
encoder = TextEmbedding(
    model_name=MODEL_NAME,
    specific_model_path=model_path,
    threads=2,
    providers=["CPUExecutionProvider"],
)
print(f"Embedding {len(products)} products on CPU in batches of 32...")
embeddings = list(
    encoder.passage_embed(
        (
            item["product_name"] + ". " + item["product_description"]
            for item in products
        ),
        batch_size=32,
    )
)
query_vector = next(iter(encoder.query_embed(QUERY_TEXT))).tolist()

COLLECTION = "wands_pickup"
if await client.collection_exists(COLLECTION):
    await client.delete_collection(COLLECTION)
await client.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size=len(query_vector), distance=models.Distance.COSINE
    ),
)
for field, schema in [
    ("in_stock", models.PayloadSchemaType.BOOL),
    ("free_returns", models.PayloadSchemaType.BOOL),
    ("pickup_location", models.PayloadSchemaType.GEO),
]:
    await client.create_payload_index(COLLECTION, field, field_schema=schema, wait=True)
await client.upsert(
    collection_name=COLLECTION,
    points=[
        models.PointStruct(
            id=item["id"],
            vector=embedding.tolist(),
            payload={key: value for key, value in item.items() if key != "id"},
        )
        for item, embedding in zip(products, embeddings, strict=True)
    ],
    wait=True,
)
print(
    f"Indexed {(await client.count(COLLECTION, exact=True)).count} points, {len(query_vector)} dimensions"
)

Embedding 1123 products on CPU in batches of 32...
Indexed 1123 points, 384 dimensions


Fetching 7 files: 100%|##########| 7/7 [00:08<00:00,  1.17s/it]


## 5. Inspect Semantic Retrieval

Retrieve products by text similarity before applying operational rules. Inspect the product names, available descriptions, and labels: a high similarity score is not a guarantee of a useful desk. Some WANDS descriptions are empty, including the first two results in the recorded run. For those products the embedding uses only the name, and the printed description is blank. Do not invent missing product specifications. Cosine scores are not probabilities.

Exact vector search over this small judged pool makes the ranking experiment easier to inspect. The initial call retrieves every indexed product so the explanation can look up its original semantic score. Production retrieval needs an evaluated candidate budget and suitable ANN settings.

In [5]:
import math


SHOPPER_LOCATION = models.GeoPoint(lat=52.2297, lon=21.0122)
CANDIDATE_LIMIT = len(products)
TOP_K = 5
EXACT = models.SearchParams(exact=True)


def distance_m(origin: models.GeoPoint, destination: dict[str, float]) -> float:
    lat1, lat2 = math.radians(origin.lat), math.radians(destination["lat"])
    dlat = lat2 - lat1
    dlon = math.radians(destination["lon"] - origin.lon)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    return 6_371_000 * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def name_of(point: models.ScoredPoint) -> str:
    assert point.payload is not None
    return f"{point.id}: {str(point.payload['product_name'])[:55]}"


def show_hits(
    points: list[models.ScoredPoint], origin: models.GeoPoint = SHOPPER_LOCATION
) -> pd.DataFrame:
    rows = []
    for point in points:
        payload = point.payload or {}
        location = payload.get("pickup_location")
        rows.append(
            {
                "id": point.id,
                "product_name": str(payload["product_name"])[:60],
                "label": label_by_id[point.id],
                "score": point.score,
                "distance_km": distance_m(origin, location) / 1000
                if location
                else None,
                "in_stock": payload["in_stock"],
                "free_returns": payload["free_returns"],
            }
        )
    return pd.DataFrame(rows).round(4)


baseline = (
    await client.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        search_params=EXACT,
        limit=CANDIDATE_LIMIT,
        with_payload=True,
    )
).points
semantic_scores = {point.id: point.score for point in baseline}
display(show_hits(baseline[:TOP_K]))
for point in baseline[:2]:
    assert point.payload is not None
    print(name_of(point), "| WANDS:", label_by_id[point.id])
    print(point.payload["product_description"], "\n")

2075: writing desk | WANDS: Exact
 

18125: lynnsy writing desk | WANDS: Exact
 



,id,product_name,label,score,distance_km,in_stock,free_returns
0,2075,writing desk,Exact,0.9497,6.5158,True,False
1,18125,lynnsy writing desk,Exact,0.8889,11.6263,True,False
2,204,brookshire writing desk,Exact,0.8840,9.3882,True,True
3,19916,leifsson writing desk,Exact,0.8799,9.3882,True,False
4,18777,dells writing desk,Exact,0.8799,1.8063,True,True


## 6. Apply Stock and Geographic Requirements

Only offers marked `in_stock` and within **15 km** of the shopper are eligible. Put both conditions in a filter. Neither a strong text match nor a large bonus can rescue an out-of-stock or distant offer.

Offers without coordinates fail the geographic filter before scoring. The radius measures geographic distance, not driving distance, delivery coverage, or travel time.

In [6]:
MAX_DISTANCE_M = 15_000.0


def make_eligibility_filter(origin: models.GeoPoint) -> models.Filter:
    return models.Filter(
        must=[
            models.FieldCondition(key="in_stock", match=models.MatchValue(value=True)),
            models.FieldCondition(
                key="pickup_location",
                geo_radius=models.GeoRadius(center=origin, radius=MAX_DISTANCE_M),
            ),
        ]
    )


eligibility_filter = make_eligibility_filter(SHOPPER_LOCATION)
filtered = (
    await client.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        query_filter=eligibility_filter,
        search_params=EXACT,
        limit=CANDIDATE_LIMIT,
        with_payload=True,
    )
).points
display(show_hits(filtered[:TOP_K]))
print(f"Eligible candidates: {len(filtered)} / {len(products)}")
assert all(isinstance(point.id, int) and point.id % 37 != 0 for point in filtered)

Eligible candidates: 812 / 1123


,id,product_name,label,score,distance_km,in_stock,free_returns
0,2075,writing desk,Exact,0.9497,6.5158,True,False
1,18125,lynnsy writing desk,Exact,0.8889,11.6263,True,False
2,204,brookshire writing desk,Exact,0.8840,9.3882,True,True
3,19916,leifsson writing desk,Exact,0.8799,9.3882,True,False
4,18777,dells writing desk,Exact,0.8799,1.8063,True,True


## 7. Prefer Free Returns

The shopper is willing to pay for a return, but prefers offers with free returns. A matching Boolean condition contributes 1 and a non-match contributes 0. Add `0.08 × free_returns` to the prefetched similarity score.

The filter belongs inside the prefetch so the candidate budget is spent on eligible offers. Formula queries only rescore those candidates; they do not retrieve a new product.

In [7]:
RETURNS_WEIGHT = 0.08
returns_condition = models.FieldCondition(
    key="free_returns", match=models.MatchValue(value=True)
)
returns_bonus = models.MultExpression(mult=[RETURNS_WEIGHT, returns_condition])

returns_boost = (
    await client.query_points(
        collection_name=COLLECTION,
        prefetch=models.Prefetch(
            query=query_vector,
            filter=eligibility_filter,
            params=EXACT,
            limit=CANDIDATE_LIMIT,
        ),
        query=models.FormulaQuery(
            formula=models.SumExpression(sum=["$score", returns_bonus])
        ),
        limit=TOP_K,
        with_payload=True,
    )
).points
display(show_hits(returns_boost))

,id,product_name,label,score,distance_km,in_stock,free_returns
0,204,brookshire writing desk,Exact,0.9640,9.3882,True,True
1,18777,dells writing desk,Exact,0.9599,1.8063,True,True
2,2075,writing desk,Exact,0.9497,6.5158,True,False
3,32394,balido writing desk with hutch,Exact,0.9272,2.6704,True,True
4,11055,harcrest writing desk,Exact,0.9207,7.0352,True,True


## 8. Prefer Closer Pickup With Geographic Decay

A radius is a hard boundary. Inside it, Gaussian decay gives a gradual preference: 1 at the shopper, 0.5 at 5 km, and 0.0625 at 10 km. `scale=5000` is where the decay reaches its midpoint, not another cutoff.

Multiplying the decay by `GEO_WEIGHT=0.15` limits its contribution to 0.15 score units. These are illustrative weights; inspect the score distribution and evaluate them for your own catalog.

In [8]:
GEO_WEIGHT = 0.15
DISTANCE_SCALE_M = 5_000.0


def make_proximity(origin: models.GeoPoint) -> models.GaussDecayExpression:
    return models.GaussDecayExpression(
        gauss_decay=models.DecayParamsExpression(
            x=models.GeoDistance(
                geo_distance=models.GeoDistanceParams(
                    origin=origin, to="pickup_location"
                )
            ),
            target=0.0,
            scale=DISTANCE_SCALE_M,
            midpoint=0.5,
        )
    )


proximity = make_proximity(SHOPPER_LOCATION)
proximity_boost = (
    await client.query_points(
        collection_name=COLLECTION,
        prefetch=models.Prefetch(
            query=query_vector,
            filter=eligibility_filter,
            params=EXACT,
            limit=CANDIDATE_LIMIT,
        ),
        query=models.FormulaQuery(
            formula=models.SumExpression(
                sum=[
                    "$score",
                    models.MultExpression(mult=[GEO_WEIGHT, proximity]),
                ]
            )
        ),
        limit=TOP_K,
        with_payload=True,
    )
).points
display(show_hits(proximity_boost))

,id,product_name,label,score,distance_km,in_stock,free_returns
0,18777,dells writing desk,Exact,1.0169,1.8063,True,True
1,29401,boland writing desk,Exact,1.0091,1.8063,True,False
2,2075,writing desk,Exact,0.9959,6.5158,True,False
3,36689,anka reversible corner writing desk with hutch,Exact,0.9839,1.8063,True,False
4,24264,mcdonough table with drawer writing desk,Exact,0.9824,0.6456,True,True


## 9. Combine and Explain the Contributions

The formula is `cosine_similarity + 0.08 × free_returns + 0.15 × proximity`.

Keep `$score` in the sum to retain the text-relevance signal. The helper lets you change the shopper, weights, and candidate limit. The component table independently reconstructs the bonuses and checks them against Qdrant's actual returned score.

In [9]:
async def rank_products(
    origin: models.GeoPoint = SHOPPER_LOCATION,
    returns_weight: float = RETURNS_WEIGHT,
    geo_weight: float = GEO_WEIGHT,
    candidate_limit: int = CANDIDATE_LIMIT,
    limit: int = TOP_K,
) -> list[models.ScoredPoint]:
    response = await client.query_points(
        collection_name=COLLECTION,
        prefetch=models.Prefetch(
            query=query_vector,
            filter=make_eligibility_filter(origin),
            params=EXACT,
            limit=candidate_limit,
        ),
        query=models.FormulaQuery(
            formula=models.SumExpression(
                sum=[
                    "$score",
                    models.MultExpression(mult=[returns_weight, returns_condition]),
                    models.MultExpression(mult=[geo_weight, make_proximity(origin)]),
                ]
            )
        ),
        limit=limit,
        with_payload=True,
    )
    return response.points


combined = await rank_products()
display(show_hits(combined))

,id,product_name,label,score,distance_km,in_stock,free_returns
0,18777,dells writing desk,Exact,1.0969,1.8063,True,True
1,24264,mcdonough table with drawer writing desk,Exact,1.0624,0.6456,True,True
2,12456,sunnyvale manufactured wood writing desk,Exact,1.0620,0.6456,True,True
3,32394,balido writing desk with hutch,Exact,1.0502,2.6704,True,True
4,9000,genowefa wall mounted 39.4 '' w writing desk,Exact,1.0362,0.6456,True,True


In [10]:
def explain_scores(
    points: list[models.ScoredPoint],
    origin: models.GeoPoint = SHOPPER_LOCATION,
    returns_weight: float = RETURNS_WEIGHT,
    geo_weight: float = GEO_WEIGHT,
) -> pd.DataFrame:
    rows = []
    for point in points:
        payload = point.payload or {}
        distance = distance_m(origin, payload["pickup_location"])
        geo_bonus = geo_weight * math.exp(
            math.log(0.5) * (distance / DISTANCE_SCALE_M) ** 2
        )
        returns_bonus = returns_weight * float(payload["free_returns"])
        reconstructed = semantic_scores[point.id] + returns_bonus + geo_bonus
        assert math.isclose(reconstructed, point.score, abs_tol=2e-5)
        rows.append(
            {
                "id": point.id,
                "product_name": str(payload["product_name"])[:60],
                "label": label_by_id[point.id],
                "semantic": semantic_scores[point.id],
                "distance_km": distance / 1000,
                "returns_bonus": returns_bonus,
                "geo_bonus": geo_bonus,
                "final_score": point.score,
            }
        )
    return pd.DataFrame(rows).round(4)


display(explain_scores(combined))
comparison = pd.DataFrame(
    {
        "rank": range(1, TOP_K + 1),
        "semantic + eligibility": [name_of(p) for p in filtered[:TOP_K]],
        "returns bonus": [name_of(p) for p in returns_boost],
        "geo bonus": [name_of(p) for p in proximity_boost],
        "combined": [name_of(p) for p in combined],
    }
)
display(comparison)

,id,product_name,label,semantic,distance_km,returns_bonus,geo_bonus,final_score
0,18777,dells writing desk,Exact,0.8799,1.8063,0.08,0.1370,1.0969
1,24264,mcdonough table with drawer writing desk,Exact,0.8342,0.6456,0.08,0.1483,1.0624
2,12456,sunnyvale manufactured wood writing desk,Exact,0.8337,0.6456,0.08,0.1483,1.0620
3,32394,balido writing desk with hutch,Exact,0.8472,2.6704,0.08,0.1231,1.0502
4,9000,genowefa wall mounted 39.4 '' w writing desk,Exact,0.8079,0.6456,0.08,0.1483,1.0362


,rank,semantic + eligibility,returns bonus,geo bonus,combined
0,1,2075: writing desk,204: brookshire writing desk,18777: dells writing desk,18777: dells writing desk
1,2,18125: lynnsy writing desk,18777: dells writing desk,29401: boland writing desk,24264: mcdonough table with drawer writing desk
2,3,204: brookshire writing desk,2075: writing desk,2075: writing desk,12456: sunnyvale manufactured wood writing desk
3,4,19916: leifsson writing desk,32394: balido writing desk with hutch,36689: anka reversible corner writing desk wit...,32394: balido writing desk with hutch
4,5,18777: dells writing desk,11055: harcrest writing desk,24264: mcdonough table with drawer writing desk,9000: genowefa wall mounted 39.4 '' w writing ...


A changed order proves that the formula changed the ranking. It does not prove that shoppers prefer the result. Free returns and pickup distance are separate from WANDS' product-relevance labels; the audit later checks whether those preferences displace relevant products.

## 10. Check the Boundaries

Zero bonuses should recover the eligible products and their semantic scores in descending order. Some WANDS products have identical text and tied scores; their relative order is not significant. Every rescored offer must still be in stock and inside the radius, and missing coordinates must stay excluded. An empty geographic region must return no recommendation. These checks verify the mechanism, not the quality of the business policy.

In [11]:
without_bonuses = await rank_products(
    returns_weight=0.0, geo_weight=0.0, limit=CANDIDATE_LIMIT
)
expected_scores = {point.id: point.score for point in filtered}
assert {point.id for point in without_bonuses} == set(expected_scores)
for point in without_bonuses:
    assert math.isclose(point.score, expected_scores[point.id], abs_tol=1e-6)
assert all(
    first.score >= second.score
    for first, second in zip(without_bonuses, without_bonuses[1:])
)
for point in await rank_products(limit=CANDIDATE_LIMIT):
    payload = point.payload or {}
    assert payload["in_stock"] is True
    assert distance_m(SHOPPER_LOCATION, payload["pickup_location"]) <= MAX_DISTANCE_M
    assert isinstance(point.id, int) and point.id % 37 != 0
empty_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="pickup_location",
            geo_radius=models.GeoRadius(center=SHOPPER_LOCATION, radius=1.0),
        )
    ]
)
empty = (
    await client.query_points(
        collection_name=COLLECTION,
        prefetch=models.Prefetch(
            query=query_vector, filter=empty_filter, limit=CANDIDATE_LIMIT
        ),
        query=models.FormulaQuery(
            formula=models.SumExpression(sum=["$score", returns_bonus])
        ),
        limit=TOP_K,
    )
).points
assert empty == []
print(
    "Passed: zero bonuses, eligibility, missing coordinates, score arithmetic, and empty result"
)

Passed: zero bonuses, eligibility, missing coordinates, score arithmetic, and empty result


## 11. Stress the Ranking Policy

**Overweight proximity.** Increase the proximity coefficient to 5 while keeping retrieval and eligibility fixed. Inspect the winner's title, description, and later human relevance grade. A larger final score means the formula assigned a larger number; it is not a quality metric. Close but less relevant products can displace better matches.

In [12]:
overweighted = await rank_products(geo_weight=5.0)
display(explain_scores(overweighted, geo_weight=5.0))
assert overweighted[0].payload is not None
print("Winner's description:", overweighted[0].payload["product_description"])

Winner's description: 


,id,product_name,label,semantic,distance_km,returns_bonus,geo_bonus,final_score
0,24264,mcdonough table with drawer writing desk,Exact,0.8342,0.6456,0.08,4.9425,5.8567
1,12456,sunnyvale manufactured wood writing desk,Exact,0.8337,0.6456,0.08,4.9425,5.8563
2,9000,genowefa wall mounted 39.4 '' w writing desk,Exact,0.8079,0.6456,0.08,4.9425,5.8304
3,8832,47 '' writing desk,Exact,0.7711,0.6456,0.08,4.9425,5.7937
4,6264,coldfield solid wood writing desk,Exact,0.7696,0.6456,0.08,4.9425,5.7921


**Retrieve too few candidates.** Compare a prefetch of only three semantic matches with a prefetch covering the whole judged pool. The same formula can only promote a product if retrieval included it. The exhaustive pool is a diagnostic reference for this example; it is not a recommended production prefetch size.

In [13]:
narrow = await rank_products(candidate_limit=3, limit=3)
wide = await rank_products(candidate_limit=CANDIDATE_LIMIT, limit=3)
display(
    pd.DataFrame(
        {
            "rank": [1, 2, 3],
            "prefetch limit 3": [name_of(p) for p in narrow],
            f"prefetch limit {CANDIDATE_LIMIT}": [name_of(p) for p in wide],
        }
    )
)
omitted = [
    name_of(p) for p in wide if p.id not in {candidate.id for candidate in narrow}
]
print("Promoted products the narrow retrieval missed:", omitted)
if not omitted:
    print("Both candidate budgets returned the same top results for this request.")

Promoted products the narrow retrieval missed: ['18777: dells writing desk', '24264: mcdonough table with drawer writing desk', '12456: sunnyvale manufactured wood writing desk']


,rank,prefetch limit 3,prefetch limit 1123
0,1,2075: writing desk,18777: dells writing desk
1,2,204: brookshire writing desk,24264: mcdonough table with drawer writing desk
2,3,18125: lynnsy writing desk,12456: sunnyvale manufactured wood writing desk


## 12. Audit Product Relevance With WANDS Labels

Map Exact to grade 2, Partial to 1, and Irrelevant to 0. NDCG@5 discounts lower ranks and uses gains `2**grade - 1`. Its ideal ranking comes from **the same eligible judged pool** for this query, so filtering does not change the denominator between these ranking policies. Every retrieved product has a judgment.

Compare eligible semantic retrieval, the combined formula, and excessive proximity weighting. Print the labels as well as the aggregate metric. WANDS judgments concern the product/query match, not our invented pickup distance, stock, or returns. This is a single-query, pool-restricted diagnostic, not a catalog-wide relevance benchmark, a held-out evaluation, or evidence that boosting improves conversion. Keep independent queries for tuning and evaluation before choosing production weights.

In the recorded November 25, 2026 run, all three top-five lists contain only Exact judgments, so all three receive NDCG@5 of 1.0. The boosted rankings change, but this query and these coarse labels do not distinguish their product relevance. The large geographic coefficient shows score domination, not measured relevance deterioration or improvement.

In [14]:
GRADE = {"Exact": 2, "Partial": 1, "Irrelevant": 0}


def dcg(grades: list[int]) -> float:
    return sum(
        (2**grade - 1) / math.log2(rank + 2) for rank, grade in enumerate(grades)
    )


eligible_grades = sorted(
    (GRADE[label_by_id[point.id]] for point in filtered), reverse=True
)
ideal_dcg = dcg(eligible_grades[:TOP_K])
audit_rows = []
for policy, hits in [
    ("semantic + eligibility", filtered[:TOP_K]),
    ("combined", combined),
    ("proximity weight 5", overweighted),
]:
    grades = [GRADE[label_by_id[point.id]] for point in hits]
    audit_rows.append(
        {
            "policy": policy,
            "NDCG@5 (eligible judged pool)": dcg(grades) / ideal_dcg
            if ideal_dcg
            else None,
            "labels in rank order": [label_by_id[point.id] for point in hits],
        }
    )
display(pd.DataFrame(audit_rows).round(4))

,policy,NDCG@5 (eligible judged pool),labels in rank order
0,semantic + eligibility,1.0,"[Exact, Exact, Exact, Exact, Exact]"
1,combined,1.0,"[Exact, Exact, Exact, Exact, Exact]"
2,proximity weight 5,1.0,"[Exact, Exact, Exact, Exact, Exact]"


## 13. Move the Shopper

Move north without re-embedding the query. Both the hard eligibility boundary and the distance contribution change. Predict which products should benefit, run the cell, and explain their score components. Change `NEW_ORIGIN` to explore another point.

The previous relevance audit used the original shopper's eligible pool. If you compare policies after moving, recompute that pool and its ideal DCG for the new origin.

In [15]:
NEW_ORIGIN = models.GeoPoint(lat=52.310, lon=21.040)
moved = await rank_products(origin=NEW_ORIGIN)
if not moved:
    print("No eligible products near this shopper. Try another origin or radius.")
else:
    display(explain_scores(moved, origin=NEW_ORIGIN))
    print("Original winner:", name_of(combined[0]))
    print("New-origin winner:", name_of(moved[0]))
    if [p.id for p in moved] == [p.id for p in combined]:
        print("The ranking is unchanged; inspect the distance contributions.")

Original winner: 18777: dells writing desk
New-origin winner: 204: brookshire writing desk


,id,product_name,label,semantic,distance_km,returns_bonus,geo_bonus,final_score
0,204,brookshire writing desk,Exact,0.8840,0.3017,0.08,0.1496,1.1136
1,9588,imogene 31.5 '' w writing desk and chair set,Exact,0.8076,0.3017,0.08,0.1496,1.0372
2,19916,leifsson writing desk,Exact,0.8799,0.3017,0.00,0.1496,1.0296
3,612,abbiss l-shape desk,Exact,0.7883,0.3017,0.08,0.1496,1.0179
4,14292,gympie writing desk,Exact,0.7881,0.3017,0.08,0.1496,1.0177


## 14. Tune the Weights Explicitly

Pass trial weights to both ranking and explanation so the displayed components match the executed formula. Python captures defaults when a function is defined; changing a global weight alone does not update those defaults.

The proximity helper reads `DISTANCE_SCALE_M` on each call, so you can change the decay scale and rerun this cell. To use another search request with matching labels, choose one of the larger example pools listed in the WANDS cell, change `QUERY_ID`, and rerun every following cell.

In [16]:
trial_returns_weight = 0.05
trial_geo_weight = 0.10
trial = await rank_products(
    returns_weight=trial_returns_weight,
    geo_weight=trial_geo_weight,
)
display(
    explain_scores(
        trial,
        returns_weight=trial_returns_weight,
        geo_weight=trial_geo_weight,
    )
)

,id,product_name,label,semantic,distance_km,returns_bonus,geo_bonus,final_score
0,18777,dells writing desk,Exact,0.8799,1.8063,0.05,0.0914,1.0213
1,24264,mcdonough table with drawer writing desk,Exact,0.8342,0.6456,0.05,0.0989,0.9830
2,12456,sunnyvale manufactured wood writing desk,Exact,0.8337,0.6456,0.05,0.0989,0.9826
3,2075,writing desk,Exact,0.9497,6.5158,0.00,0.0308,0.9805
4,32394,balido writing desk with hutch,Exact,0.8472,2.6704,0.05,0.0821,0.9792


## Extend to Amazon ESCI or a Production Catalog

The [Amazon ESCI embeddings bucket](https://huggingface.co/buckets/itsclelia/esci-products-embeddings) offers a larger pre-embedded e-commerce path. Download it separately with the Hugging Face CLI command `hf buckets sync hf://buckets/itsclelia/esci-products-embeddings ./storage` after checking current size and available disk space. This download is optional and is not executed by the notebook.

The bucket README inspected on November 25, 2026 describes 5,781,746 products, with dense/BM25, additional CLIP, neural sparse, and ColBERT files with different coverage. Its dense vectors have 384 dimensions, but the inspected README does not identify their model. Dimensionality alone does not establish compatibility with this notebook's BGE vectors: confirm the encoder and preprocessing before embedding new queries. Join representations using the documented product IDs, inspect each folder's metadata, and configure the appropriate dense, sparse, or multivector fields.

The bucket's 2,000 held-out query titles are titles of products, not the human-labeled WANDS search queries or a verified ESCI relevance-judgment set. They cannot substitute for the labels used in this notebook's audit. The bucket contents and size may change; the extension has not been executed here.

For either dataset, obtain maintained operational metadata separately. Neither a relevant product description nor a relevance label establishes local availability. Index offer-level coordinates and stock, deduplicate products when several stores carry them, and keep hard requirements in filters. Evaluate candidate recall, ranking metrics, and latency over representative held-out queries. Recalibrate weights after changing the embedding model, distance metric, or introducing hybrid retrieval; score scales differ.

References: [WANDS](https://github.com/wayfair/WANDS), [score boosting and decay](https://qdrant.tech/documentation/search/search-relevance/), [multi-stage queries](https://qdrant.tech/documentation/search/hybrid-queries/), [geographic filters](https://qdrant.tech/documentation/search/filtering/), [payload indexes](https://qdrant.tech/documentation/manage-data/indexing/), [pinned server](https://github.com/qdrant/qdrant/releases/tag/v1.20.2), and [pinned model](https://huggingface.co/qdrant/bge-small-en-v1.5-onnx-q/tree/aa8f8b060edb00e03bfdd08813a2949946c8ba55).

## Optional Cleanup

Set `CLEAN_UP=True` after experimenting. This stops the notebook's server and removes only its temporary directory. The server also terminates when the kernel exits.

In [17]:
CLEAN_UP = False
if CLEAN_UP:
    await client.close()
    server.terminate()
    server.wait(timeout=10)
    server_log.close()
    shutil.rmtree(runtime_dir)
    print("Stopped the tutorial server and removed its temporary data")